In [ ]:
pip install tensorflow_addons

In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import pandas as pd

# =======================
# 1. 配置参数
# =======================
data_dir = '/kaggle/input/ucmerced-landuse/UCMerced_LandUse/Images'
num_classes = 21
batch_size = 32
num_epochs = 30
lr = 3e-3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 早停设置
patience = 7
delta = 0.0

# =======================
# 2. 数据预处理
# =======================
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root=data_dir)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [train_size, val_size]
)

class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = TransformedSubset(
    torch.utils.data.Subset(full_dataset, train_indices),
    transform=transform_train
)
val_dataset = TransformedSubset(
    torch.utils.data.Subset(full_dataset, val_indices),
    transform=transform_val
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# =======================
# 3. 模型定义函数（无预训练）
# =======================
def get_model(model_name):
    if model_name == 'efficientnet_b0':
        return timm.create_model('efficientnet_b0', pretrained=False, num_classes=num_classes)
    elif model_name == 'mobilenetv3_small_100':
        return timm.create_model('mobilenetv3_small_100', pretrained=False, num_classes=num_classes)
    elif model_name == 'rexnet_100':
        return timm.create_model('rexnet_100', pretrained=False, num_classes=num_classes)
    elif model_name == 'ghostnet_100':
        return timm.create_model('ghostnet_100', pretrained=False, num_classes=num_classes)
    else:
        raise ValueError(f"Unknown model: {model_name}")

# =======================
# 4. 早停类
# =======================
class EarlyStopping:
    def __init__(self, patience=7, delta=0.0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_acc):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0

# =======================
# 5. 评估函数
# =======================
def evaluate_loader(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return correct / total

# =======================
# 6. 训练函数
# =======================
def train_model(model, model_name, train_loader, val_loader, num_epochs, patience):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    early_stopping = EarlyStopping(patience=patience, delta=delta)
    best_acc = 0.0

    for epoch in range(num_epochs):
        # ----- 训练 -----
        model.train()
        running_loss = 0.0
        correct_train, total_train = 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        train_acc = correct_train / total_train

        # ----- 验证 -----
        val_acc = evaluate_loader(model, val_loader)

        print(f"[{model_name}] Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {running_loss/len(train_loader):.4f} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        # 保存最佳模型
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'best_{model_name}_from_scratch.pth')
            print(f"[{model_name}] ✅ New best validation accuracy: {best_acc:.4f}, model saved!")

        # 早停检查
        early_stopping(val_acc)
        if early_stopping.early_stop:
            print(f"[{model_name}] 🛑 Early stopping at epoch {epoch+1}")
            break

    return best_acc

# =======================
# 7. 运行实验
# =======================
results = []
model_names = ['efficientnet_b0', 'mobilenetv3_small_100', 'rexnet_100', 'ghostnet_100']

for name in model_names:
    print(f"\n{'='*50}")
    print(f"     Training {name.upper()} (from scratch)")
    print(f"{'='*50}")
    model = get_model(name).to(device)
    params = sum(p.numel() for p in model.parameters()) / 1e6
    best_val_acc = train_model(model, name, train_loader, val_loader, num_epochs, patience)
    results.append({'Model': name.upper(), 'Best Val Accuracy': best_val_acc, 'Params (M)': round(params, 2)})

# =======================
# 8. 输出结果
# =======================
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='Best Val Accuracy', ascending=False)
print("\n" + "="*70)
print("📊 MODEL PERFORMANCE SUMMARY (From Scratch + Early Stopping)")
print("="*70)
print(df_results.to_string(index=False))

df_results.to_csv('results_from_scratch_earlystop.csv', index=False)
print("\n✅ Results saved to 'results_from_scratch_earlystop.csv'")



     Training EFFICIENTNET_B0 (from scratch)
[efficientnet_b0] Epoch 1/30 | Train Loss: 3.4354 | Train Acc: 0.1524 | Val Acc: 0.2119
[efficientnet_b0] ✅ New best validation accuracy: 0.2119, model saved!
[efficientnet_b0] Epoch 2/30 | Train Loss: 2.2581 | Train Acc: 0.2679 | Val Acc: 0.2500
[efficientnet_b0] ✅ New best validation accuracy: 0.2500, model saved!
[efficientnet_b0] Epoch 3/30 | Train Loss: 2.0904 | Train Acc: 0.2869 | Val Acc: 0.3452
[efficientnet_b0] ✅ New best validation accuracy: 0.3452, model saved!
[efficientnet_b0] Epoch 4/30 | Train Loss: 1.7962 | Train Acc: 0.4179 | Val Acc: 0.4048
[efficientnet_b0] ✅ New best validation accuracy: 0.4048, model saved!
[efficientnet_b0] Epoch 5/30 | Train Loss: 1.5848 | Train Acc: 0.4810 | Val Acc: 0.4881
[efficientnet_b0] ✅ New best validation accuracy: 0.4881, model saved!
[efficientnet_b0] Epoch 6/30 | Train Loss: 1.4728 | Train Acc: 0.5060 | Val Acc: 0.4643
[efficientnet_b0] Epoch 7/30 | Train Loss: 1.3964 | Train Acc: 0.5446 |